# 02_RQ1_DGRAPHFIN_v2_1

**Trương Minh Hoàng · MSSV 25410008**

Notebook nghiên cứu KLTN v2.1. Đọc [hướng dẫn chạy](README.md) và chuẩn bị dữ liệu trước khi thực thi. Output Colab cũ được bỏ; kết quả đã khóa nằm ở [evidence/historical](../evidence/historical/).

Research notebook: dataset-backed execution is separate from the small public demonstrations. Scientific feature formulas, model candidates and routing checks are retained. Public packaging changes cover paths, optional installation and fresh-run defaults.

# 02 — RQ1 DGraphFin: node-only vs node + graph-derived features (v2.1)

Thiết kế chính: cùng LightGBM, cùng train/validation/test masks, khác duy nhất nhóm graph-derived features đã khai báo.

**Hai pha bắt buộc:**

1. `FINAL_MODE=False`: chỉ train/validation, tạo `rq1_freeze_manifest.json`; **không đọc `y[test]`, không tính test metric**.
2. Sau khi kiểm tra freeze, đặt trong notebook 00: `FINAL_MODE=True` và `FINAL_TEST_AUTHORISED=True`, lưu config, rồi chạy notebook 02 từ đầu. Final mode tải đúng model/config đã freeze và mở test một lần.

Graph feature setting là **transductive về topology/node features nhưng không dùng test labels**. Kết quả không được gọi là dự báo tương lai.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# Install the optional research requirements before running.
# %pip -q install "torch-geometric>=2.5,<3" "lightgbm>=4.3,<5" pyarrow scipy scikit-learn psutil

In [ ]:
import os
from IPython.display import display
from pathlib import Path
import gc, hashlib, json, os, random, sys, time
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

POINTER_PATH = Path(os.environ.get('KLTN_CONFIG_POINTER', '/content/drive/MyDrive/KLTN_REPRODUCTION_CONFIG_POINTER.json' if Path('/content/drive/MyDrive').is_dir() else str(Path.cwd() / 'KLTN_REPRODUCTION_CONFIG_POINTER.json'))).resolve()
if not POINTER_PATH.exists():
    raise FileNotFoundError('Chưa có config pointer. Hãy chạy notebook 00 trước.')
CONFIG_PATH = Path(json.loads(POINTER_PATH.read_text(encoding='utf-8'))['config_path'])
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

RQ1_DIR = Path(CONFIG['rq1_dir']); RQ1_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(CONFIG['cache_dir']); CACHE_DIR.mkdir(parents=True, exist_ok=True)
DGRAPH_ROOT = Path(CONFIG['dgraph_root'])
DGRAPH_ZIP = Path(CONFIG['dgraph_zip'])
FINAL_MODE = bool(CONFIG.get('final_mode', False))
FINAL_TEST_AUTHORISED = bool(CONFIG.get('final_test_authorised', False))
TECHNICAL_RERUN_OVERRIDE = bool(CONFIG.get('technical_rerun_override', False))
PRIMARY_SEED = int(CONFIG.get('primary_seed', 42))
STABILITY_SEEDS = [int(s) for s in CONFIG.get('stability_seeds', [])]
GRAPH_VISIBILITY_POLICY = CONFIG['graph_visibility_policy']

random.seed(PRIMARY_SEED); np.random.seed(PRIMARY_SEED)
print({'final_mode': FINAL_MODE, 'final_test_authorised': FINAL_TEST_AUTHORISED, 'seed': PRIMARY_SEED, 'visibility': GRAPH_VISIBILITY_POLICY})


In [ ]:
import lightgbm as lgb
import scipy.sparse as sp
import torch
import torch_geometric
from torch_geometric.datasets import DGraphFin
from sklearn.metrics import average_precision_score, roc_auc_score

if not DGRAPH_ZIP.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu: {DGRAPH_ZIP}')

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(obj: dict) -> str:
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def mask_to_index(mask) -> np.ndarray:
    values = mask.cpu().numpy() if hasattr(mask, 'cpu') else np.asarray(mask)
    values = values.reshape(-1)
    return np.flatnonzero(values) if values.dtype == bool else values.astype('int64')

def labeled_only(index: np.ndarray, labels: np.ndarray) -> np.ndarray:
    return index[np.isin(labels[index], [0, 1])]

dataset_sha256 = sha256_file(DGRAPH_ZIP)
dataset = DGraphFin(root=str(DGRAPH_ROOT))
data = dataset[0]
x = data.x.cpu().numpy().astype('float32', copy=False)
y = data.y.cpu().numpy().reshape(-1)
edge_index = data.edge_index.cpu().numpy().astype('int64', copy=False)
edge_type = data.edge_type.cpu().numpy().reshape(-1).astype('int64', copy=False) if hasattr(data, 'edge_type') else np.zeros(edge_index.shape[1], dtype='int64')

train_mask_idx = mask_to_index(data.train_mask)
val_mask_idx = mask_to_index(data.val_mask)
test_mask_idx = mask_to_index(data.test_mask)  # chỉ chỉ mục; chưa đọc y[test]
train_idx = labeled_only(train_mask_idx, y)
val_idx = labeled_only(val_mask_idx, y)
for name, idx in [('train', train_idx), ('validation', val_idx)]:
    labels, counts = np.unique(y[idx], return_counts=True)
    print(name, len(idx), dict(zip(labels.astype(int), counts.astype(int))))
    if set(labels.tolist()) != {0, 1}:
        raise ValueError(f'{name} không có đủ hai lớp 0/1.')
print({'test_mask_count_without_label_inspection': int(len(test_mask_idx)), 'dataset_sha256': dataset_sha256})


In [ ]:
if len(edge_type) and int(edge_type.min()) < 0:
    raise ValueError('edge_type contains negative values; feature encoding assumes non-negative relation types.')

GRAPH_FEATURE_SPEC = {
    'version': 'dgraphfin_graph_features_v2.1',
    'visibility_policy': GRAPH_VISIBILITY_POLICY,
    'uses_labels': False,
    'features': [
        'log1p_in_degree', 'log1p_out_degree', 'reciprocal_ratio',
        'log1p_out_relation_type_diversity', 'out_neighbor_mean_node_features'
    ],
    'duplicate_edge_policy': 'binary_adjacency_after_sum_duplicates',
    'isolated_node_policy': 'degree_zero_and_neighbor_mean_zero',
    'direction': 'directed_out_neighbors_for_neighbor_means',
}
feature_spec_sha256 = canonical_sha256(GRAPH_FEATURE_SPEC)
cache_path = CACHE_DIR / 'dgraphfin_graph_features_v2_1.npy'
cache_manifest_path = CACHE_DIR / 'dgraphfin_graph_features_v2_1_manifest.json'

def build_graph_features(x: np.ndarray, edge_index: np.ndarray, edge_type: np.ndarray) -> np.ndarray:
    n_nodes = x.shape[0]
    src, dst = edge_index
    adjacency = sp.csr_matrix((np.ones(len(src), dtype='float32'), (src, dst)), shape=(n_nodes, n_nodes))
    adjacency.sum_duplicates(); adjacency.data[:] = 1.0

    out_degree = np.asarray(adjacency.sum(axis=1)).ravel().astype('float32')
    in_degree = np.asarray(adjacency.sum(axis=0)).ravel().astype('float32')
    reciprocal_degree = np.asarray(adjacency.multiply(adjacency.T).getnnz(axis=1)).ravel().astype('float32')
    reciprocal_ratio = reciprocal_degree / np.maximum(out_degree, 1.0)

    type_base = int(edge_type.max()) + 1 if len(edge_type) else 1
    unique_source_type = np.unique(src.astype('int64') * type_base + edge_type.astype('int64'))
    unique_sources = unique_source_type // type_base
    relation_diversity = np.bincount(unique_sources, minlength=n_nodes).astype('float32')

    neighbor_sum = adjacency @ x
    neighbor_mean = neighbor_sum / np.maximum(out_degree[:, None], 1.0)
    result = np.column_stack([
        np.log1p(in_degree), np.log1p(out_degree), reciprocal_ratio,
        np.log1p(relation_diversity), neighbor_mean,
    ]).astype('float32')
    return result

cache_ok = False
if cache_path.exists() and cache_manifest_path.exists():
    old_manifest = json.loads(cache_manifest_path.read_text(encoding='utf-8'))
    cache_ok = (
        old_manifest.get('dataset_sha256') == dataset_sha256 and
        old_manifest.get('feature_spec_sha256') == feature_spec_sha256 and
        old_manifest.get('x_shape') == list(x.shape) and
        old_manifest.get('edge_index_shape') == list(edge_index.shape) and
        old_manifest.get('cache_sha256') == sha256_file(cache_path)
    )

if cache_ok:
    graph_features = np.load(cache_path, mmap_mode='r')
    print('CACHE PASS:', graph_features.shape)
else:
    if cache_path.exists() or cache_manifest_path.exists():
        print('Cache cũ không khớp manifest; tính lại.')
    graph_features = build_graph_features(x, edge_index, edge_type)
    np.save(cache_path, graph_features)
    cache_manifest = {
        'dataset_sha256': dataset_sha256,
        'feature_spec_sha256': feature_spec_sha256,
        'feature_spec': GRAPH_FEATURE_SPEC,
        'x_shape': list(x.shape),
        'edge_index_shape': list(edge_index.shape),
        'graph_feature_shape': list(graph_features.shape),
        'cache_sha256': sha256_file(cache_path),
    }
    cache_manifest_path.write_text(json.dumps(cache_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã tạo cache:', graph_features.shape)

expected_cols = 4 + x.shape[1]
if list(graph_features.shape) != [x.shape[0], expected_cols]:
    raise ValueError(f'Graph feature shape không đúng: {graph_features.shape}, expected {(x.shape[0], expected_cols)}')

graph_feature_names = ['log1p_in_degree', 'log1p_out_degree', 'reciprocal_ratio', 'log1p_out_relation_type_diversity'] + [f'out_neighbor_mean_x{i}' for i in range(x.shape[1])]

# Giảm peak RAM trước khi tạo ma trận LightGBM.
del edge_index, edge_type, data, dataset
gc.collect()


In [ ]:
def matrix(index: np.ndarray, include_graph: bool) -> np.ndarray:
    if include_graph:
        return np.column_stack([x[index], np.asarray(graph_features[index])]).astype('float32')
    return x[index].astype('float32', copy=False)

def score_metrics(y_true: np.ndarray, score: np.ndarray, entity_ids: np.ndarray) -> dict:
    top_n = max(1, int(np.ceil(0.01 * len(score))))
    order = np.lexsort((entity_ids.astype('int64'), -score))[:top_n]
    return {
        'average_precision': float(average_precision_score(y_true, score)),
        'roc_auc': float(roc_auc_score(y_true, score)),
        'top_1_percent_n': int(top_n),
        'top_1_percent_positive': int(y_true[order].sum()),
    }

CANDIDATES = [
    {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100},
    {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 200},
    {'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 500},
]
y_train = y[train_idx].astype('int8')
y_val = y[val_idx].astype('int8')
scale_pos_weight = int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1)

X_train_base = matrix(train_idx, False); X_val_base = matrix(val_idx, False)
X_train_graph = matrix(train_idx, True); X_val_graph = matrix(val_idx, True)

def tune_variant(name, X_train, X_val):
    rows, fitted = [], []
    for candidate_id, candidate in enumerate(CANDIDATES):
        model = lgb.LGBMClassifier(
            objective='binary', n_estimators=3000, random_state=PRIMARY_SEED,
            n_jobs=-1, colsample_bytree=0.9, subsample=0.9, subsample_freq=1,
            reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
            verbosity=-1, **candidate
        )
        model.fit(
            X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='average_precision',
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
        )
        val_score = model.predict_proba(X_val, num_iteration=model.best_iteration_)[:, 1]
        rows.append({
            'variant': name, 'candidate_id': candidate_id,
            'best_iteration': int(model.best_iteration_),
            **score_metrics(y_val, val_score, val_idx), **candidate,
        })
        fitted.append(model)
    table = pd.DataFrame(rows).sort_values(['average_precision', 'candidate_id'], ascending=[False, True])
    best_row = table.iloc[0]
    best_id = int(best_row['candidate_id'])
    return table, fitted[best_id], CANDIDATES[best_id]


In [ ]:
FREEZE_PATH = RQ1_DIR / 'rq1_freeze_manifest.json'
FINAL_MANIFEST_PATH = RQ1_DIR / 'rq1_final_manifest.json'
FINAL_ACCESS_PATH = RQ1_DIR / 'rq1_final_access_started.json'
PRE_ACCESS_PATH = RQ1_DIR / 'rq1_pre_access_integrity.json'
BASE_MODEL_PATH = RQ1_DIR / 'node_only_frozen_model.txt'
GRAPH_MODEL_PATH = RQ1_DIR / 'node_plus_graph_frozen_model.txt'

if not FINAL_MODE:
    if FINAL_MANIFEST_PATH.exists() and not TECHNICAL_RERUN_OVERRIDE:
        raise RuntimeError('Final evidence đã tồn tại. Không được ghi đè freeze sau final nếu chưa có technical override.')

    base_tuning, base_model, base_params = tune_variant('node_only', X_train_base, X_val_base)
    graph_tuning, graph_model, graph_params = tune_variant('node_plus_graph', X_train_graph, X_val_graph)
    tuning = pd.concat([base_tuning, graph_tuning], ignore_index=True)
    tuning.to_csv(RQ1_DIR / 'rq1_validation_tuning.csv', index=False)
    display(tuning)

    base_model.booster_.save_model(str(BASE_MODEL_PATH))
    graph_model.booster_.save_model(str(GRAPH_MODEL_PATH))

    freeze = {
        'status': 'RQ1_FREEZE_READY',
        'protocol_version': CONFIG['protocol_version'],
        'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'dataset_sha256': dataset_sha256,
        'graph_visibility_policy': GRAPH_VISIBILITY_POLICY,
        'feature_spec_sha256': feature_spec_sha256,
        'graph_cache_sha256': sha256_file(cache_path),
        'graph_cache_manifest_sha256': sha256_file(cache_manifest_path),
        'graph_feature_names': graph_feature_names,
        'primary_metric': 'average_precision',
        'secondary_metrics': ['roc_auc', 'top_1_percent_positive'],
        'primary_seed': PRIMARY_SEED,
        'stability_seeds': STABILITY_SEEDS,
        'candidate_grid': CANDIDATES,
        'node_only': {
            'selected_params': base_params,
            'best_iteration': int(base_model.best_iteration_),
            'model_sha256': sha256_file(BASE_MODEL_PATH),
        },
        'node_plus_graph': {
            'selected_params': graph_params,
            'best_iteration': int(graph_model.best_iteration_),
            'model_sha256': sha256_file(GRAPH_MODEL_PATH),
        },
        'environment': {
            'python': sys.version,
            'numpy': np.__version__,
            'lightgbm': lgb.__version__,
            'torch': torch.__version__,
            'torch_geometric': torch_geometric.__version__,
        },
        'test_labels_accessed': False,
        'claim_boundary': 'DGraphFin benchmark-level incremental value of declared graph-derived features; transductive topology/node-feature visibility; no prospective deployment claim',
    }
    FREEZE_PATH.write_text(json.dumps(freeze, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(freeze, ensure_ascii=False, indent=2))
    print('\nDEVELOPMENT COMPLETE — TEST CHƯA MỞ. Kiểm tra freeze manifest trước khi bật FINAL_MODE + FINAL_TEST_AUTHORISED trong notebook 00.')
else:
    print('FINAL_MODE=True: bỏ qua tuning; sẽ dùng đúng freeze manifest/model đã lưu.')


In [ ]:
print("PRE_ACCESS exists:", PRE_ACCESS_PATH.exists())
print("FINAL_ACCESS exists:", FINAL_ACCESS_PATH.exists())
print("FINAL_MANIFEST exists:", FINAL_MANIFEST_PATH.exists())

In [ ]:
# === FINAL TEST GATE v2.1.1 — PATCHED ===
# Mục tiêu:
# 1) Không truy cập y_test trước khi mọi integrity check PASS.
# 2) Ghi PRE_ACCESS record trước.
# 3) Ghi FINAL_ACCESS_STARTED record sau đó.
# 4) Chỉ sau khi FINAL_ACCESS_STARTED đã được persist mới đọc test labels.
# 5) Không có duplicate block / circular hash dependency.

if FINAL_MODE:
    RQ1_DIR.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------
    # A. FINAL-AUTHORITY / RERUN GUARDS
    # ------------------------------------------------------------------
    if not FINAL_TEST_AUTHORISED:
        raise RuntimeError('REFUSED: FINAL_TEST_AUTHORISED=False.')

    if not FREEZE_PATH.exists():
        raise FileNotFoundError(
            'REFUSED: chưa có rq1_freeze_manifest.json từ development run.'
        )

    if FINAL_MANIFEST_PATH.exists() and not TECHNICAL_RERUN_OVERRIDE:
        raise RuntimeError(
            'REFUSED: final manifest đã tồn tại; '
            'không chạy lại final nếu không có technical override.'
        )

    if FINAL_ACCESS_PATH.exists() and not TECHNICAL_RERUN_OVERRIDE:
        raise RuntimeError(
            'REFUSED: final test đã từng bắt đầu truy cập; '
            'nếu lần trước lỗi kỹ thuật phải bật TECHNICAL_RERUN_OVERRIDE '
            'và ghi lý do ngoài notebook.'
        )

    # ------------------------------------------------------------------
    # B. VERIFY FROZEN AUTHORITY — CHƯA ĐỌC TEST LABELS
    # ------------------------------------------------------------------
    freeze = json.loads(FREEZE_PATH.read_text(encoding='utf-8'))

    if freeze.get('status') != 'RQ1_FREEZE_READY':
        raise RuntimeError('Freeze status không hợp lệ.')

    if freeze.get('dataset_sha256') != dataset_sha256:
        raise RuntimeError('Dataset hash khác freeze.')

    if freeze.get('feature_spec_sha256') != feature_spec_sha256:
        raise RuntimeError('Feature spec khác freeze.')

    if freeze.get('graph_visibility_policy') != GRAPH_VISIBILITY_POLICY:
        raise RuntimeError('Graph visibility policy khác freeze.')

    if not cache_path.exists():
        raise FileNotFoundError('Graph feature cache không tồn tại.')

    if not cache_manifest_path.exists():
        raise FileNotFoundError('Graph cache manifest không tồn tại.')

    if not BASE_MODEL_PATH.exists():
        raise FileNotFoundError('Node-only frozen model không tồn tại.')

    if not GRAPH_MODEL_PATH.exists():
        raise FileNotFoundError('Node+graph frozen model không tồn tại.')

    if sha256_file(cache_path) != freeze.get('graph_cache_sha256'):
        raise RuntimeError('Graph feature cache hash khác freeze.')

    if sha256_file(cache_manifest_path) != freeze.get('graph_cache_manifest_sha256'):
        raise RuntimeError('Graph cache manifest hash khác freeze.')

    if sha256_file(BASE_MODEL_PATH) != freeze['node_only']['model_sha256']:
        raise RuntimeError('Node-only model hash khác freeze.')

    if sha256_file(GRAPH_MODEL_PATH) != freeze['node_plus_graph']['model_sha256']:
        raise RuntimeError('Graph model hash khác freeze.')

    # ------------------------------------------------------------------
    # C. PRE-ACCESS INTEGRITY RECORD
    # ------------------------------------------------------------------
    # Không tự băm chính PRE_ACCESS_PATH trước khi file được tạo.
    # Record này xác nhận mọi frozen artifact đã PASS trước test access.
    pre_access = {
        'status': 'RQ1_PRE_ACCESS_INTEGRITY_PASS',
        'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'freeze_manifest_sha256': sha256_file(FREEZE_PATH),
        'dataset_sha256': dataset_sha256,
        'feature_spec_sha256': feature_spec_sha256,
        'graph_visibility_policy': GRAPH_VISIBILITY_POLICY,
        'graph_cache_sha256': sha256_file(cache_path),
        'graph_cache_manifest_sha256': sha256_file(cache_manifest_path),
        'node_only_model_sha256': sha256_file(BASE_MODEL_PATH),
        'node_plus_graph_model_sha256': sha256_file(GRAPH_MODEL_PATH),
        'final_test_authorised': bool(FINAL_TEST_AUTHORISED),
        'technical_rerun_override': bool(TECHNICAL_RERUN_OVERRIDE),
        'test_labels_accessed_at_record_creation': False,
    }

    PRE_ACCESS_PATH.write_text(
        json.dumps(pre_access, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

    pre_access_sha256 = sha256_file(PRE_ACCESS_PATH)

    print('PRE-ACCESS INTEGRITY RECORD:', PRE_ACCESS_PATH)
    print('PRE-ACCESS SHA256:', pre_access_sha256)

    # ------------------------------------------------------------------
    # D. FINAL ACCESS START RECORD
    # ------------------------------------------------------------------
    # Từ thời điểm record này được persist, lần final được xem là đã bắt đầu.
    final_access = {
        'status': 'FINAL_ACCESS_STARTED',
        'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'freeze_manifest_sha256': sha256_file(FREEZE_PATH),
        'pre_access_integrity_sha256': pre_access_sha256,
        'final_test_authorised': bool(FINAL_TEST_AUTHORISED),
        'technical_rerun_override': bool(TECHNICAL_RERUN_OVERRIDE),
        'note': (
            'Test labels may only be accessed after this record has been persisted.'
        ),
    }

    FINAL_ACCESS_PATH.write_text(
        json.dumps(final_access, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

    final_access_sha256 = sha256_file(FINAL_ACCESS_PATH)

    print('FINAL ACCESS RECORD:', FINAL_ACCESS_PATH)
    print('FINAL ACCESS SHA256:', final_access_sha256)

    # ==================================================================
    # E. FINAL TEST ACCESS BOUNDARY
    # CHỈ TỪ DÒNG NÀY MỚI ĐƯỢC TRUY CẬP TEST LABELS.
    # ==================================================================
    test_idx = labeled_only(test_mask_idx, y)
    y_test = y[test_idx].astype('int8')

    labels, counts = np.unique(y_test, return_counts=True)
    if set(labels.tolist()) != {0, 1}:
        raise ValueError('Test labeled subset không có đủ hai lớp 0/1.')

    print(
        'FINAL TEST LABEL COUNTS:',
        dict(zip(labels.astype(int), counts.astype(int)))
    )

    # ------------------------------------------------------------------
    # F. PRIMARY FROZEN MODELS — ONE FINAL EVALUATION
    # ------------------------------------------------------------------
    X_test_base = matrix(test_idx, False)
    X_test_graph = matrix(test_idx, True)

    base_booster = lgb.Booster(model_file=str(BASE_MODEL_PATH))
    graph_booster = lgb.Booster(model_file=str(GRAPH_MODEL_PATH))

    base_test_score = base_booster.predict(X_test_base)
    graph_test_score = graph_booster.predict(X_test_graph)

    primary_results = pd.DataFrame([
        {
            'variant': 'node_only',
            'seed': PRIMARY_SEED,
            **score_metrics(y_test, base_test_score, test_idx),
        },
        {
            'variant': 'node_plus_graph',
            'seed': PRIMARY_SEED,
            **score_metrics(y_test, graph_test_score, test_idx),
        },
    ])

    delta_ap = float(
        primary_results.loc[
            primary_results['variant'].eq('node_plus_graph'),
            'average_precision'
        ].iloc[0]
        -
        primary_results.loc[
            primary_results['variant'].eq('node_only'),
            'average_precision'
        ].iloc[0]
    )

    primary_metrics_path = RQ1_DIR / 'rq1_primary_metrics.csv'
    primary_results.to_csv(primary_metrics_path, index=False)

    display(primary_results)
    print('Delta AP:', delta_ap)

    # ------------------------------------------------------------------
    # G. CONTEXT PREDICTIONS / LABELS — TÁCH RIÊNG
    # ------------------------------------------------------------------
    common = {
        'source_record_reference': test_idx.astype(str),
        'entity_scope': 'dgraphfin_user_node',
        'risk_domain': 'fraud_graph',
        'score_semantics': 'dgraphfin_anomaly_class_score',
        'source_dataset': 'dgraphfin_original',
        'assessment_time': [None] * len(test_idx),
        'assessment_time_status': 'not_available_in_source',
    }

    node_only_context = pd.DataFrame({
        **common,
        'score_value': base_test_score.astype('float64'),
        'model_version': 'dgraphfin_lightgbm_node_only_v2.1',
    })

    graph_context = pd.DataFrame({
        **common,
        'score_value': graph_test_score.astype('float64'),
        'model_version': 'dgraphfin_lightgbm_node_plus_graph_v2.1',
    })

    labels_df = pd.DataFrame({
        'source_record_reference': test_idx.astype(str),
        'y_true': y_test,
    })

    node_path = RQ1_DIR / 'dgraphfin_node_only_predictions_for_context.parquet'
    graph_path = RQ1_DIR / 'dgraphfin_node_plus_graph_predictions_for_context.parquet'
    label_path = RQ1_DIR / 'dgraphfin_evaluation_labels.parquet'

    node_only_context.to_parquet(node_path, index=False)
    graph_context.to_parquet(graph_path, index=False)
    labels_df.to_parquet(label_path, index=False)

    # ------------------------------------------------------------------
    # H. SEED STABILITY — SAME FROZEN PARAMS, NO RE-TUNING
    # ------------------------------------------------------------------
    def refit_frozen(X_train, X_test, selected_params, n_estimators, seed):
        model = lgb.LGBMClassifier(
            objective='binary',
            n_estimators=int(n_estimators),
            random_state=int(seed),
            n_jobs=-1,
            colsample_bytree=0.9,
            subsample=0.9,
            subsample_freq=1,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            verbosity=-1,
            **selected_params,
        )
        model.fit(
            X_train,
            y_train,
            callbacks=[lgb.log_evaluation(0)],
        )
        return model.predict_proba(X_test)[:, 1]

    stability_rows = primary_results.to_dict('records')

    for seed in STABILITY_SEEDS:
        b = refit_frozen(
            X_train_base,
            X_test_base,
            freeze['node_only']['selected_params'],
            freeze['node_only']['best_iteration'],
            seed,
        )

        g = refit_frozen(
            X_train_graph,
            X_test_graph,
            freeze['node_plus_graph']['selected_params'],
            freeze['node_plus_graph']['best_iteration'],
            seed,
        )

        stability_rows.append({
            'variant': 'node_only',
            'seed': seed,
            **score_metrics(y_test, b, test_idx),
        })

        stability_rows.append({
            'variant': 'node_plus_graph',
            'seed': seed,
            **score_metrics(y_test, g, test_idx),
        })

    stability = pd.DataFrame(stability_rows)
    stability_path = RQ1_DIR / 'rq1_seed_stability.csv'
    stability.to_csv(stability_path, index=False)

    # ------------------------------------------------------------------
    # I. PAIRED BOOTSTRAP — FROZEN AT 10,000 EVALUABLE RESAMPLES
    # ------------------------------------------------------------------
    # Lưu ý: đây là node-level paired bootstrap.
    # Phụ thuộc qua graph vẫn là limitation và được ghi trong summary.
    rng = np.random.default_rng(2026)
    deltas = []
    n = len(y_test)

    requested_evaluable = 10_000
    skipped_single_class = 0

    while len(deltas) < requested_evaluable:
        sample = rng.integers(0, n, size=n)
        ys = y_test[sample]

        if np.unique(ys).size < 2:
            skipped_single_class += 1
            continue

        deltas.append(
            average_precision_score(
                ys,
                graph_test_score[sample],
            )
            -
            average_precision_score(
                ys,
                base_test_score[sample],
            )
        )

    deltas = np.asarray(deltas, dtype='float64')

    bootstrap_path = RQ1_DIR / 'rq1_paired_bootstrap_10000.npy'
    np.save(bootstrap_path, deltas)

    if deltas.size != requested_evaluable:
        raise RuntimeError(
            f'Bootstrap incomplete: {deltas.size}/{requested_evaluable}.'
        )

    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])

    interpretation = (
        'POSITIVE_INDICATION'
        if ci_low > 0
        else (
            'NEGATIVE_INDICATION'
            if ci_high < 0
            else 'NO_STABLE_DIRECTION'
        )
    )

    summary = {
        'status': 'RQ1_FINAL_COMPLETE',
        'primary_seed': int(PRIMARY_SEED),
        'test_rows': int(len(y_test)),
        'delta_ap': float(delta_ap),
        'delta_ap_ci_95': [float(ci_low), float(ci_high)],
        'bootstrap_requested_evaluable': int(requested_evaluable),
        'bootstrap_evaluable': int(len(deltas)),
        'bootstrap_skipped_single_class': int(skipped_single_class),
        'bootstrap_seed': 2026,
        'classification': interpretation,
        'interpretation_scope': (
            'paired node resampling; graph dependence remains a limitation'
        ),
    }

    summary_path = RQ1_DIR / 'rq1_summary.json'
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    # ------------------------------------------------------------------
    # J. FINAL EVIDENCE MANIFEST
    # ------------------------------------------------------------------
    final_manifest = {
        'status': 'RQ1_FINAL_EVIDENCE_LOCKED',
        'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'freeze_manifest_sha256': sha256_file(FREEZE_PATH),
        'pre_access_integrity_sha256': sha256_file(PRE_ACCESS_PATH),
        'final_access_record_sha256': sha256_file(FINAL_ACCESS_PATH),
        'technical_rerun_override_used': bool(TECHNICAL_RERUN_OVERRIDE),
        'files': {
            node_path.name: sha256_file(node_path),
            graph_path.name: sha256_file(graph_path),
            label_path.name: sha256_file(label_path),
            primary_metrics_path.name: sha256_file(primary_metrics_path),
            stability_path.name: sha256_file(stability_path),
            summary_path.name: sha256_file(summary_path),
            bootstrap_path.name: sha256_file(bootstrap_path),
        },
        'summary': summary,
    }

    FINAL_MANIFEST_PATH.write_text(
        json.dumps(final_manifest, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print('FINAL EVIDENCE LOCKED:', FINAL_MANIFEST_PATH)

else:
    print(
        'FINAL TEST SKIPPED — development mode giữ test labels chưa truy cập.'
    )
